In [ ]:
!pip install datasets evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import transformers
import torch
from datasets import Dataset,load_dataset
import evaluate
!kaggle datasets download namanj27/ner-dataset
!unzip /content/ner-dataset.zip
df=pd.read_csv("/content/ner_datasetreference.csv", encoding='latin1')
def lower_w(w):
  if type(w)!=float:
    return w.lower()
  else:
    return w
df['Word']=df['Word'].apply(lower_w)
Sentences=[]
words=[]
ners=[]
current_sentence_key = None
for c, row in df.iterrows():
  if not pd.isna(row['Sentence #']):
    current_sentence_key = row['Sentence #']
  if current_sentence_key is not None:
    Sentences.append(current_sentence_key)
    words.append(row['Word'])
    ners.append(row['Tag'])
new=np.vstack([Sentences,words,ners])
new_df=pd.DataFrame(new.T)

from sklearn.preprocessing import LabelEncoder
lbl=LabelEncoder()
lbl.fit(new_df[2])
new_df[3]=lbl.transform(new_df[2])
label2id={}
id2label={}
for c,i in enumerate(lbl.classes_):
  label2id[c]=i
  id2label[i]=c
sents=new_df.groupby(0)
sents_order=sents.groups.keys()

all_sent=[]
all_ner=[]
for s in sents_order:
  text=sents.get_group(s)[1].values
  all_sent.append(text)
  ner=sents.get_group(s)[3].values
  all_ner.append(ner)

final_df=pd.DataFrame([all_sent,all_ner])
final_df=final_df.T
final_df.columns=['words','labels']
from datasets import ClassLabel, Sequence

ds=Dataset.from_pandas(final_df)

sorted_label_names = [label2id[i] for i in sorted(label2id.keys())]
single_label_feature = ClassLabel(names=sorted_label_names)
sequence_label_feature = Sequence(feature=single_label_feature)

ds = ds.cast_column("labels", sequence_label_feature)

ds=ds.train_test_split(test_size=0.2,shuffle=True)
ds['train'].features['labels'].feature
from transformers import AutoTokenizer, AutoModelForTokenClassification

tokenizer = AutoTokenizer.from_pretrained("dslim/bert-base-NER")

def algo(sampels):
  new_ner=[-100]
  for n,w in zip(sampels['labels'],sampels['words']):
    word_token=tokenizer.tokenize(w)
    frist_seen=True
    for i in word_token:
      n_str=label2id[n]
      if frist_seen:
        new_ner.append(id2label[n_str])
        frist_seen=False
      else:
        if n_str!="O":
          new_ner.append(id2label["I-"+n_str.split("-")[1]])
        else:
          new_ner.append(id2label[n_str])
  new_ner.append(-100)
  sampels["new_ner"] = new_ner
  return sampels

new_ds=ds.map(algo)
new_ds=new_ds.remove_columns(['labels'])
new_ds=new_ds.rename_column("new_ner","labels")
def word_tokeniz(sampels):
  return tokenizer(sampels['words'],is_split_into_words=True)
new_ds=new_ds.map(word_tokeniz,remove_columns=['words'])
new_ds=new_ds.with_format("torch")
from transformers import DataCollatorForTokenClassification

data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer)
from torch.utils.data import DataLoader

batches=DataLoader(new_ds['train'],batch_size=8, collate_fn=data_collator)
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained("dslim/bert-base-NER",num_labels=17,label2id=id2label,id2label=label2id,ignore_mismatched_sizes=True)
model.config._num_labels=17
from transformers import TrainingArguments,Trainer

training_arg=TrainingArguments(output_dir="/content/model_check",
                  per_device_eval_batch_size=16,
                  per_device_train_batch_size=16,
                  num_train_epochs=5,
                  fp16 =True,
                  save_strategy="epoch",
                  warmup_steps=500
                  )
trainer=Trainer(
    model=model,
    args=training_arg,
    train_dataset=new_ds['train'],
    eval_dataset=new_ds['test'],
    data_collator=data_collator
)
trainer.train()
trainer.save_model("/content/my_model")
from transformers import AutoTokenizer, AutoModelForTokenClassification

my_tokenizer = AutoTokenizer.from_pretrained("/content/my_model")

my_model=AutoModelForTokenClassification.from_pretrained("/content/my_model")
sent="My name is mo salah i work in elnasr company".lower()
res=my_tokenizer(sent.split(" "),is_split_into_words=True,return_tensors="pt")
outputs=my_model(**res)
out=np.squeeze(outputs.logits.detach().numpy())
lbls=[]
for row in out:
  lbls.append(label2id[np.argmax(row)])
for l,t in zip(lbls,my_tokenizer.convert_ids_to_tokens(res.input_ids[0])):
  print((t,l))
from transformers import pipeline

pipe = pipeline("token-classification", model="/content/my_model")

In [ ]:
!pip install datasets evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import transformers
import torch
from datasets import Dataset,load_dataset
import evaluate
from datasets import load_dataset

ds = load_dataset("mustapha/QuranExe")
ds=ds['train']
ds=ds.remove_columns(['resource_name', 'verses_keys'])
from transformers import AutoTokenizer, AutoModelForMaskedLM

tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
new_tokenizer=tokenizer.train_new_from_iterator(ds['text'],30522)
model = AutoModelForMaskedLM.from_pretrained("google-bert/bert-base-uncased")
def toknization_and_pad(exampels):
  return new_tokenizer(exampels['text'],truncation=True)
new_ds=ds.map(toknization_and_pad)
new_ds=new_ds.remove_columns(["text"])
new_ds=new_ds.train_test_split(test_size=0.3)
from transformers import DataCollatorForLanguageModeling

data_collator=DataCollatorForLanguageModeling(new_tokenizer,mlm=True,mlm_probability=0.15,return_tensors="pt")
from transformers import TrainingArguments,Trainer

training_arg=TrainingArguments(output_dir="/content/model_check",
                  per_device_eval_batch_size=16,
                  per_device_train_batch_size=16,
                  num_train_epochs=5,
                  fp16 =True,
                  save_strategy="epoch",
                  warmup_steps=500,
                  )
trainer=Trainer(
    model=model,
    args=training_arg,
    train_dataset=new_ds['train'].select(range(1000)),
    eval_dataset=new_ds['test'].select(range(1000)),
    data_collator=data_collator,
)

trainer.train()

In [ ]:
# !pip install datasets evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import transformers
import torch
from datasets import Dataset,load_dataset
# import evaluate

df = pd.read_parquet("hf://datasets/MBZUAI-Paris/DODa-10K/data/train-00000-of-00001.parquet")
df=df[df.columns[[2,4]]]
def clean_eng_txt(txt):

  txt=txt.lower()
  txt=txt.strip()
  return txt
df['English']=df['English'].apply(clean_eng_txt)
ds=Dataset.from_pandas(df)
ds=ds.train_test_split(test_size=0.2)
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

model_checkpoint = "Helsinki-NLP/opus-mt-en-ar"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensors="pt")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
en_sentence = ds["train"][0]["English"]
ar_sentence = ds["train"][0]["MSA"]

inputs = tokenizer(en_sentence, text_target=ar_sentence)
inputs
max_length = 128
def preprocess_function(examples):
    inputs = examples['English']
    targets = examples["MSA"]
    model_inputs = tokenizer(
        inputs, text_target=targets, max_length=max_length, truncation=True
    )
    return model_inputs
tokenized_datasets = ds.map(
    preprocess_function,
    batched=True,
    remove_columns=ds["train"].column_names,
)
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
batch = data_collator([tokenized_datasets["train"][0]])
batch['labels']
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    "salah-finetuned-en-to-ar",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
)
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)
trainer.train()
trainer.save_model("/content/my_model")
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM

model_name = "/content/my_model"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tok = tokenizer('He added, "We now have 4-month-old mice that previously had diabetes, but no longer have it."', return_tensors="pt").input_ids
output = model.generate(tok)[0]
tokenizer.decode(output, skip_special_tokens=True)


In [ ]:
# !pip install datasets evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import transformers
import torch
from datasets import Dataset,load_dataset
# import evaluate
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad")
ds=ds.remove_columns(['id', 'title'])
def get_start_end_idx(batch_of_features):
  start_indices = []
  end_indices = []
  for answers_for_one_example in batch_of_features['answers']:
    answer = answers_for_one_example['text'][0]
    answer_start = answers_for_one_example['answer_start'][0]
    answer_end = answer_start + len(answer)
    start_indices.append(answer_start)
    end_indices.append(answer_end)
  return {'start_idx': start_indices, 'end_idx': end_indices}
ds=ds.map(get_start_end_idx,batched=True,remove_columns=['answers'])
ds=ds['validation']
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

tokenizer = AutoTokenizer.from_pretrained("deepset/roberta-base-squad2")
model = AutoModelForQuestionAnswering.from_pretrained("deepset/roberta-base-squad2")
def tokenization(exampels):
  tokens=tokenizer(
      exampels['question'],
      exampels['context'],
      return_offsets_mapping=True,
      truncation="only_second"
  )
  return tokens
ds=ds.map(tokenization)
def get_start_end_idx_token_based(exampels):
  for c,i in enumerate(exampels['offset_mapping']):
    new_start_idx=0
    new_end_idx=0
    start_idx=i[0]
    end_idx=i[1]
    if start_idx==exampels['start_idx']:
      new_start_idx=c
      exampels['new_start_idx']=new_start_idx
    if end_idx==exampels['end_idx']:
      new_end_idx=c+1
      exampels['new_end_idx']=new_end_idx
  return exampels
df=Dataset.to_pandas(ds)
new_df=df.apply(get_start_end_idx_token_based, axis=1)
new_ds=Dataset.from_pandas(new_df)

new_ds=new_ds.remove_columns(['context', 'end_idx','offset_mapping','question','start_idx'])
dataset = new_ds.rename_column("new_start_idx","start_positions")
dataset = dataset.rename_column("new_end_idx","end_positions")
dataset=dataset.with_format("torch")
# Explicitly cast to torch.long to ensure correct type for CrossEntropyLoss
def ensure_long_tensors(examples):
    examples['start_positions'] = examples['start_positions'].long()
    examples['end_positions'] = examples['end_positions'].long()
    return examples

dataset = dataset.map(ensure_long_tensors, batched=True)
dataset=dataset.train_test_split(test_size=0.2)
from transformers import DataCollatorWithPadding

data_collator=DataCollatorWithPadding(tokenizer)

from transformers import TrainingArguments,Trainer

training_arg=TrainingArguments(output_dir="/content/model_checks",
                  per_device_eval_batch_size=16,
                  per_device_train_batch_size=16,
                  num_train_epochs=3,
                  save_strategy="epoch",
                  warmup_steps=500,
                  eval_strategy="epoch",
                  )

trainer=Trainer(
    model=model,
    args=training_arg,
    train_dataset=dataset['train'].select(range(4000)),
    eval_dataset=dataset['test'].select(range(1000)),
    data_collator=data_collator
)


trainer.train()
trainer.save_model("/content/my_model")

my_tokenizer = AutoTokenizer.from_pretrained("/content/my_model")
my_model = AutoModelForQuestionAnswering.from_pretrained("/content/my_model")

ds2 = load_dataset("rajpurkar/squad")
ds2=ds2.remove_columns(['id', 'title'])
s=ds2['train'][40000]
s
toks=my_tokenizer(s['question'],s['context'],return_tensors="pt")
out=my_model(**toks)
start_tok=out.start_logits.argmax()
end_tok=out.end_logits.argmax()
(start_tok,end_tok)
my_tokenizer.decode(toks['input_ids'][0][start_tok:end_tok]).strip()

In [ ]:
# !pip install datasets evaluate
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import keras
import transformers
import torch
from datasets import Dataset,load_dataset
# import evaluate
from datasets import load_dataset
import re
!kaggle datasets download abdallahelsaadany/fatawa
!unzip /content/fatawa.zip
df = pd.read_csv('/content/88kData.csv')
df = df.drop('ID', axis=1)
df = df.dropna()
ll = []
ll+=list(df[df['ans'].str.contains('انظر إلى الفتوى')].index)
ll+=list(df[df['ans'].str.contains('رقم:')].index)
ll+=list(df[df['ans'].str.contains('فتوى رقم')].index)
df = df.drop(index=ll, axis=0)
df = df.drop(index=84488, axis=0)

def extract_arabic_and_english_numerics(text):
    arabic_pattern = r'[^a-zA-Z\s0-9]+'
    numeric_pattern = r'\d+'
    combined_pattern = f'({arabic_pattern}|{numeric_pattern})'

    matches = re.findall(combined_pattern, text)

    extracted_text = ' '.join(matches)

    return extracted_text

df['ans'] = df['ans'].apply(extract_arabic_and_english_numerics)
df['ques'] = df['ques'].apply(extract_arabic_and_english_numerics)
df['title'] = df['title'].apply(extract_arabic_and_english_numerics)

listt=[]
for k in range(len(df)):

    if len(df['ans'].iloc[k]) < 118:
        listt.append(k)
listt = list(df.iloc[listt].index)
df = df.drop(index=listt, axis=0)
df = df.astype(str)
df.drop_duplicates(inplace=True)
df = df[0:10000]
df["full_text"]=" question: " + df["title"]+df["ques"]+" answer: "+df["ans"]

ds=Dataset.from_pandas(df)
ds=ds.remove_columns(['__index_level_0__','title','ques',"ans"])
ds=ds.train_test_split(test_size=0.25,shuffle=True)
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-base")
model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-base")
tokenizer.convert_ids_to_tokens(tokenizer.encode(ds['test'][0]['full_text']))
def toknization(exampels):
  toks=tokenizer(exampels['full_text'], truncation=True)
  return toks
new_ds=ds.map(toknization,remove_columns=['full_text'])
from transformers import DataCollatorForLanguageModeling

data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)
help(TrainingArguments)
from transformers import TrainingArguments,Trainer

training_arg=TrainingArguments(output_dir="/content/model_check",
                  per_device_eval_batch_size=4,
                  per_device_train_batch_size=4,
                  num_train_epochs=3,
                  save_strategy="epoch",
                  gradient_accumulation_steps=4,
                  warmup_steps=500,
                  )

trainer=Trainer(
    model=model,
    args=training_arg,
    train_dataset=new_ds['train'].select(range(4000)),
    eval_dataset=new_ds['test'].select(range(2000)),
    data_collator=data_collator
)


trainer.train()